# Introduction

Understanding the factors that influence happiness is a key topic in economics, sociology, and data science. With the availability of large-scale international datasets, it has become possible to quantitatively analyze how different economic and lifestyle indicators relate to subjective well-being across countries.

This project explores the relationship between happiness levels, GDP per capita, and smoking prevalence. GDP per capita is used as an indicator of economic development, while smoking prevalence represents a lifestyle and public health factor. Happiness is measured using survey-based data from international reports.

The main goal of this analysis is to investigate how these variables are related and to what extent they can explain differences in happiness across countries.

## Research Questions
- What is the relationship between smoking prevalence and happiness across countries?
- How does GDP per capita relate to happiness levels?
- To what extent do smoking prevalence and GDP per capita jointly explain variations in happiness?

## Methodology
The project follows a structured data science workflow:
- Data collection from multiple independent sources
- Data cleaning and preprocessing
- Standardization of country names and formats
- Merging datasets into a unified structure
- Exploratory data analysis (EDA)
- Correlation analysis
- Linear regression modeling

## Scope and Limitations
This study focuses on statistical relationships rather than causal effects. The analysis is conducted at the country level and does not account for individual-level differences, cultural variation, or other potential confounding factors.

Despite these limitations, the project provides useful insights into global patterns of happiness and lifestyle choices and economic conditions.


## Data Preparation and Preprocessing

Three datasets were used in this analysis:
- GDP per capita
- Smoking prevalence
- Happiness scores


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the datasets
gdp = pd.read_csv("GDP PER CAPITA.csv", sep=";", skiprows=4)
smoking = pd.read_csv("Smoking world .csv", sep=";")
happiness = pd.read_csv("World Happiness 2013-2023.csv", sep=";")

print("GDP columns:", gdp.columns.tolist()[:10], "...")
print("Smoking columns:", smoking.columns.tolist())
print("Happiness columns:", happiness.columns.tolist())

The datasets were loaded from CSV files. Initial inspection showed differences in structure, naming conventions, and numeric formatting. In particular, the GDP dataset was stored in wide format, while the happiness dataset used commas as decimal separators.

In [ ]:
# Clean and reshape GDP data
gdp.columns = gdp.columns.astype(str).str.strip()
gdp = gdp.rename(columns={"Country Name": "country"})

gdp = gdp[["country", "2020", "2021", "2022"]]
gdp = gdp.melt(
    id_vars="country",
    var_name="year",
    value_name="gdp"
)

gdp["year"] = gdp["year"].astype(int)
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")

gdp.head()

The GDP dataset was originally in wide format, with years stored as separate columns. It was reshaped into long format so that each row represents a single country-year observation.

In [ ]:
# Clean smoking data
smoking.columns = smoking.columns.str.strip()

smoking = smoking.rename(columns={
    "Entity": "country",
    "Year": "year",
    "Age-standardized prevalence of current tobacco use among persons aged 15 years and older, by sex (%) - Both sexes": "smoking"
})

smoking = smoking[["country", "year", "smoking"]]
smoking["year"] = smoking["year"].astype(int)
smoking["smoking"] = pd.to_numeric(smoking["smoking"], errors="coerce")
smoking = smoking[smoking["year"].between(2020, 2022)]

smoking.head()

The smoking dataset required column renaming and filtering to the target period 2020-2022. Only the country, year, and smoking prevalence variables were retained.

In [ ]:
# Clean happiness data
happiness.columns = happiness.columns.str.strip()

happiness = happiness.rename(columns={
    "Country": "country",
    "Year": "year",
    "Index": "happiness"
})

happiness = happiness[["country", "year", "happiness"]]
happiness["year"] = happiness["year"].astype(int)
happiness["happiness"] = happiness["happiness"].astype(str).str.replace(",", ".", regex=False)
happiness["happiness"] = pd.to_numeric(happiness["happiness"], errors="coerce")
happiness = happiness[happiness["year"].between(2020, 2022)]

happiness.head()

The happiness dataset required conversion of numeric values from text format because the decimal separator was stored as a comma. After conversion, the dataset was filtered to the same time period as the other sources.

In [ ]:
# Keep only the needed columns
gdp = gdp[["country", "year", "gdp"]]
smoking = smoking[["country", "year", "smoking"]]
happiness = happiness[["country", "year", "happiness"]]

After standardizing names and formats, only the variables relevant to the analysis were kept. This simplifies the merge process and reduces unnecessary complexity.

In [ ]:
# Merge the datasets
df = happiness.merge(gdp, on=["country", "year"], how="inner")
df = df.merge(smoking, on=["country", "year"], how="inner")
df = df.dropna()

df.head()

The three datasets were merged using country and year as common keys. Rows with missing values were removed to ensure that the final analytical dataset contains complete observations for all variables.

In [ ]:
# Final dataset check
df.info()
df.describe()

The final dataset contains country-level observations with three main variables: happiness, GDP per capita, and smoking prevalence. Due to differences in data availability across sources, some countries or years may not appear in the merged dataset.

## Exploratory Data Analysis

The next step is to explore the relationships between the main variables through visualizations and summary statistics. This helps identify patterns, detect outliers, and assess whether linear modeling is appropriate.

In [ ]:
# Scatter plots
sns.scatterplot(data=df, x="gdp", y="happiness")
plt.title("GDP per Capita vs Happiness")
plt.show()

sns.scatterplot(data=df, x="smoking", y="happiness")
plt.title("Smoking Prevalence vs Happiness")
plt.show()

In [ ]:
# Correlation matrix
corr = df[["happiness", "gdp", "smoking"]].corr()
sns.heatmap(corr, annot=True, cmap="Blues")
plt.title("Correlation Matrix")
plt.show()